# Explicación del Modelo Predictivo (XGBoost) y Explicabilidad (SHAP)

Este cuaderno tiene como objetivo demostrar el paso a paso del entrenamiento del modelo predictivo de demanda de inscripciones utilizado en la **Academia Autopoiesis**.

> **Nota de Aplicación en el Código Fuente:**
> Todo el flujo presentado en esta libreta es un reflejo pedagógico del script automatizado que se encuentra en `backend/ml/train_model.py`. En la aplicación real, cuando se ejecuta dicho script, el modelo resultante es serializado (guardado en formato `.pkl`) para ser luego utilizado por el endpoint de predicción ubicado en `backend/app.py` (ruta `/api/ia/predecir-demanda`).

## 1. Carga de Librerías y Configuración
Comenzamos importando las librerías matemáticas y de Machine Learning necesarias.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

# Inicializamos la carga interactiva de JavaScript para SHAP
shap.initjs()

## 2. Carga del Dataset de Cohortes
El algoritmo se entrena utilizando la agrupación por cohortes (`cohortes_dataset.csv`), no sobre estudiantes individuales. Esto evita ruido transaccional y nos da el *tamaño de la cohorte* (Tamano_Cohorte) que es lo que queremos predecir.

> **Enlace al código:** Equivalente a la línea 28 de `backend/ml/train_model.py`.

In [ ]:
# Cargamos el dataset
df = pd.read_csv('../data/cohortes_dataset.csv')
print(f"Dimensiones del dataset original: {df.shape} (309 Cohortes con 44 Características)")
df.head(3)

## 3. Prevención de *Data Leakage* y Separación de Variables
Se deben eliminar de los predictores ($X$) aquellos identificadores que no aportan valor matemático (como el nombre del Programa o ID de Cohorte) y, por supuesto, la variable objetivo ($y$), que es `Tamano_Cohorte`.

> **Enlace al código:** Líneas 33-35 de `backend/ml/train_model.py`.

In [ ]:
columnas_excluidas = ['Programa', 'Cohorte', 'Fecha_Primer_Inscrito', 'Tamano_Cohorte']
X = df.drop(columns=columnas_excluidas)
y = df['Tamano_Cohorte']

# División 80% Entrenamiento y 20% Prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train set: {X_train.shape[0]} cohortes | Test set: {X_test.shape[0]} cohortes")

## 4. Entrenamiento y Búsqueda de Hiperparámetros
Para garantizar que el modelo XGBoost no sufra de *overfitting* (sobreajuste), utilizamos `GridSearchCV` que aplica validación cruzada para encontrar la mejor combinación de árboles, profundidad y tasa de aprendizaje.

> **Enlace al código:** Líneas 44-64 de `backend/ml/train_model.py`.

In [ ]:
xgb_model = xgb.XGBRegressor(random_state=42, objective='reg:squarederror')

param_grid = {
    'n_estimators': [50, 100],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5]
}

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error', 
    cv=3, 
    verbose=1
)

grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_
print("\nMejores parámetros encontrados:", grid_search.best_params_)

## 5. Evaluación de Confiabilidad (Testing)
Ahora pasamos el conjunto de prueba (`X_test`) que el modelo nunca ha visto para verificar su margen de error en un escenario real.

> **Enlace al código:** Las métricas calculadas aquí (MAE, RMSE, R²) son las que se envían directamente al Frontend en la *Ficha Técnica del Modelo* (`frontend/src/pages/dashboard/FichaTecnicaModelo.jsx`).

In [ ]:
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Margen de Error (MAE): ±{mae:.2f} alumnos")
print(f"Volatilidad (RMSE): {rmse:.2f} alumnos")
print(f"Confiabilidad (R²): {r2*100:.1f}%")

## 6. Explicabilidad con SHAP (Inteligencia Artificial Explicable)
Utilizamos SHAP (SHapley Additive exPlanations) derivado de la Teoría de Juegos, para romper la "caja negra" del XGBoost y entender matemáticamente por qué el modelo toma ciertas decisiones y qué variables tienen mayor impacto en las inscripciones.

> **Enlace al código:** Líneas 86-98 de `backend/ml/train_model.py`. Este es el mismo gráfico que se guarda físicamente como `/images/ia/shap_summary.png` y se expone en la plataforma web.

In [ ]:
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

# Gráfico de Resumen (Summary Plot)
# El eje X muestra el impacto en el tamaño del cohorte.
# El color muestra si el valor original de la variable era alto (rojo) o bajo (azul).
shap.summary_plot(shap_values, X_test)